In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../data/raw/dataset.csv")
original = df.copy()

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

print("CE columns:", len(ce_cols))
print("PE columns:", len(pe_cols))
print("Total missing:", df[ce_cols + pe_cols].isna().sum().sum())

CE columns: 14
PE columns: 14
Total missing: 5460


In [2]:
filled = df.copy()

ce_row_mean = filled[ce_cols].mean(axis=1)
pe_row_mean = filled[pe_cols].mean(axis=1)

for col in ce_cols:
    filled[col] = filled[col].fillna(ce_row_mean)

for col in pe_cols:
    filled[col] = filled[col].fillna(pe_row_mean)

print("Remaining missing:", filled[ce_cols + pe_cols].isna().sum().sum())

Remaining missing: 0


In [3]:
rows = []

for idx in original.index:
    dt = original.loc[idx, "datetime"]
    
    for col in ce_cols + pe_cols:
        if pd.isna(original.loc[idx, col]):
            rows.append({
                "id": f"{dt}||{col}",
                "value": filled.loc[idx, col]
            })

submission = pd.DataFrame(rows)

print(submission.shape)
submission.head()

(5460, 2)


,id,value
0,07-01-2026 09:15||NIFTY27JAN2625500CE,0.102447
1,07-01-2026 09:15||NIFTY27JAN2625800CE,0.102447
2,07-01-2026 09:15||NIFTY27JAN2624100PE,0.143129
3,07-01-2026 09:20||NIFTY27JAN2625300CE,0.098723
4,07-01-2026 09:20||NIFTY27JAN2625400CE,0.098723


In [4]:
os.makedirs("../outputs/submissions", exist_ok=True)

submission.to_csv("../outputs/submissions/submission_mean_baseline.csv", index=False)

print("Saved successfully!")

Saved successfully!


In [5]:
sandbox = pd.read_csv("../data/raw/sandbox_solution.csv")

print(sandbox.shape)
print(sandbox.head())
print(submission.shape)
print(submission.head())

print("Same IDs:", set(submission["id"]) == set(sandbox["id"]))

(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
1  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
3  07-01-2026 09:20||NIFTY27JAN2624000PE  0.147267
4  07-01-2026 09:20||NIFTY27JAN2624200PE  0.147267
(5460, 2)
                                      id     value
0  07-01-2026 09:15||NIFTY27JAN2625500CE  0.102447
1  07-01-2026 09:15||NIFTY27JAN2625800CE  0.102447
2  07-01-2026 09:15||NIFTY27JAN2624100PE  0.143129
3  07-01-2026 09:20||NIFTY27JAN2625300CE  0.098723
4  07-01-2026 09:20||NIFTY27JAN2625400CE  0.098723
Same IDs: True
